# DeepVox Phase 3 — ASR Conformer (Codec2 → Texte Français)

**Notebook Kaggle** — GPU T4 (15.6 GB VRAM)

Pipeline :
1. Charger les dépendances (torchaudio inclus)
2. Cloner le repo DeepVox
3. Charger le pickle préprocessé 586k
4. Entraîner le modèle **Conformer-CTC** (10.1 M params)
5. Évaluer WER / CER greedy + beam search KenLM

## Architecture

- **Conformer encoder** : 14 blocs, d=176, 4 heads, conv kernel 31, ffn=704
- **Loss** : CTC pure (même que Phase 2)
- **Optimizer** : AdamW lr=1e-3 + warmup 1000 steps
- **Mixed precision** : bf16 (gain VRAM + vitesse)

Entrée : frames Codec2 (48 features / 40ms)  
Sortie : texte français (caractères, vocab=49)

**Resume** : checkpoint complet sauvegardé à chaque epoch dans `/kaggle/working/{RUN_NAME}/training_state.pth`.

## Historique des runs

| Run | Phase | Architecture | Corpus | Best CER dev | WER (KenLM) | Doc |
|---|---|---|---|---|---|---|
| #4 | 2 | BiLSTM 9.1M | 586k | 21.4% | 42.6% | `docs/15_*.md` |
| **#5** | **3** | **Conformer 10.1M** | **586k** | **cible ≤ 17%** | **cible ≤ 35%** | `docs/18_*.md` (à créer) |

Voir `docs/16_etude_comparative_bilstm_conformer.md` et `docs/17_proposition_phase3_conformer.md`.

## 0. Configuration du run

In [ ]:
# ============================================================
# CONFIGURATION DU RUN — modifier ici pour chaque expérience
# ============================================================
RUN_NAME = "run5_conformer_586k"

# Architecture Conformer (10.1 M params, 40.6 MB)
CONFORMER_D_MODEL = 176
CONFORMER_NHEAD = 4
CONFORMER_NUM_LAYERS = 14
CONFORMER_FFN_DIM = 704
CONFORMER_CONV_KERNEL = 31
CONFORMER_DROPOUT = 0.1

# Training
MAX_SAMPLES = 586_000
MAX_EPOCHS = 30
BATCH_SIZE = 24            # ↓ vs BiLSTM (32) — attention plus VRAM-hungry
LEARNING_RATE = 1e-3       # ↑ vs BiLSTM (1e-4) — Transformer optimal
WARMUP_STEPS = 1000
PATIENCE = 7
WEIGHT_DECAY = 1e-2
GRAD_CLIP = 5.0
MAX_DURATION_S = 12.0
NUM_WORKERS = 0
USE_AMP = True

# Pas de pretrained — from-scratch (architecture différente du BiLSTM)
PRETRAINED_PATH = None

# Dataset préprocessé (le même qu'en Phase 2)
PREPROCESSED_PATH = "/kaggle/input/datasets/oumarlol/deepvox-preprocessed/deepvox_586k.pkl"
# ============================================================
print(f"Run: {RUN_NAME}")
print(f"Architecture: Conformer d={CONFORMER_D_MODEL}, L={CONFORMER_NUM_LAYERS}, heads={CONFORMER_NHEAD}")
print(f"MAX_SAMPLES={MAX_SAMPLES:,}, MAX_EPOCHS={MAX_EPOCHS}, BATCH={BATCH_SIZE}")
print(f"LR={LEARNING_RATE} (warmup {WARMUP_STEPS} steps), PATIENCE={PATIENCE}")
print(f"AMP={USE_AMP}")
print(f"Preprocessed data: {PREPROCESSED_PATH}")

In [ ]:
# Installer pycodec2 (preprocessing) et vérifier torch + torchaudio (Conformer)
!pip install -q pycodec2 praatio librosa soundfile

import os
import sys
import torch
import torchaudio
print(f'PyTorch {torch.__version__}')
print(f'torchaudio {torchaudio.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    print(f'bfloat16 support: {torch.cuda.is_bf16_supported()}')

In [ ]:
! rm -rf /kaggle/working/DeepVox

In [ ]:
# Cloner le repo DeepVox
!git clone https://github.com/oumar5/DeepVox.git 2>/dev/null || echo 'Already cloned'
sys.path.insert(0, 'DeepVox/src')

from deepvox.codec2.encoder import encode_pcm, unpack_frames, SAMPLE_RATE
from deepvox.data.text import VOCAB_SIZE, BLANK_IDX, encode, decode, decode_ctc, normalize_text
from deepvox.models.conformer_asr import ConformerASR
from deepvox.eval.wer import wer, cer

print(f'Vocab size: {VOCAB_SIZE}')
print('Imports OK')

## 1. Charger les données préprocessées

In [ ]:
from pathlib import Path
import pickle
import numpy as np
from tqdm.auto import tqdm

assert PREPROCESSED_PATH and os.path.isfile(PREPROCESSED_PATH), \
    f'Preprocessed file not found: {PREPROCESSED_PATH}'

print(f'Loading preprocessed data from {PREPROCESSED_PATH}...')
with open(PREPROCESSED_PATH, 'rb') as f:
    samples = pickle.load(f)
print(f'Loaded {len(samples):,} samples')

# Stats
frame_lens = [len(s[0]) for s in samples]
char_lens = [len(s[1]) for s in samples]
print(f'Frames/sample : min={min(frame_lens)}, max={max(frame_lens)}, mean={np.mean(frame_lens):.0f}')
print(f'Chars/sample : min={min(char_lens)}, max={max(char_lens)}, mean={np.mean(char_lens):.0f}')

## 2. Dataset + DataLoader

**Important** : le Conformer demande `lengths` pour masquer le padding dans l'attention. Le `collate_fn` le retourne déjà.

In [ ]:
from torch.utils.data import Dataset, DataLoader
import random

class ASRDatasetKaggle(Dataset):
    def __init__(self, samples):
        self.samples = samples
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        feats, char_ids, _text = self.samples[idx]
        return (
            torch.from_numpy(feats).float(),
            torch.tensor(char_ids, dtype=torch.long),
            len(feats),
            len(char_ids),
        )


def ctc_collate(batch):
    feats, chars, f_lens, c_lens = zip(*batch)
    max_T = max(f_lens)
    max_L = max(c_lens)
    B = len(batch)
    feats_pad = torch.zeros(B, max_T, 48)
    chars_pad = torch.zeros(B, max_L, dtype=torch.long)
    for i in range(B):
        feats_pad[i, :feats[i].size(0)] = feats[i]
        chars_pad[i, :chars[i].size(0)] = chars[i]
    return feats_pad, chars_pad, torch.tensor(f_lens), torch.tensor(c_lens)


# Split 90/5/5 — même seed que Phase 2 pour comparabilité directe
random.seed(42)
random.shuffle(samples)
n = len(samples)
n_train = int(n * 0.9)
n_dev = int(n * 0.05)

train_ds = ASRDatasetKaggle(samples[:n_train])
dev_ds = ASRDatasetKaggle(samples[n_train:n_train+n_dev])
test_ds = ASRDatasetKaggle(samples[n_train+n_dev:])

print(f'Train: {len(train_ds)}, Dev: {len(dev_ds)}, Test: {len(test_ds)}')

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          collate_fn=ctc_collate, num_workers=NUM_WORKERS, pin_memory=True)
dev_loader = DataLoader(dev_ds, batch_size=BATCH_SIZE, shuffle=False,
                        collate_fn=ctc_collate, num_workers=NUM_WORKERS, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                         collate_fn=ctc_collate, num_workers=NUM_WORKERS, pin_memory=True)

## 3. Modèle Conformer + setup entraînement

In [ ]:
import gc

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

model = ConformerASR(
    input_dim=48,
    d_model=CONFORMER_D_MODEL,
    nhead=CONFORMER_NHEAD,
    num_layers=CONFORMER_NUM_LAYERS,
    dim_feedforward=CONFORMER_FFN_DIM,
    conv_kernel=CONFORMER_CONV_KERNEL,
    dropout=CONFORMER_DROPOUT,
    vocab_size=VOCAB_SIZE,
).to(device)

print(f'Params: {model.count_parameters():,}')
print(f'Size (float32): {model.count_parameters() * 4 / 1e6:.1f} MB')

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    betas=(0.9, 0.98),  # standard pour Transformer
    eps=1e-9,
)

# Scheduler : warmup linéaire + cosine annealing
def lr_lambda(step):
    if step < WARMUP_STEPS:
        return step / max(1, WARMUP_STEPS)
    # Cosine decay sur le reste
    progress = (step - WARMUP_STEPS) / max(1, MAX_EPOCHS * len(train_loader) - WARMUP_STEPS)
    return max(0.1, 0.5 * (1.0 + np.cos(np.pi * progress)))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
criterion = torch.nn.CTCLoss(blank=BLANK_IDX, reduction='mean', zero_infinity=True)

# AMP scaler (utile en fp16 ; bf16 n'a pas besoin de scaler)
scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP and not torch.cuda.is_bf16_supported())
amp_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print(f'AMP dtype: {amp_dtype}, scaler enabled: {scaler.is_enabled() if USE_AMP else False}')

# Resume si checkpoint existe
SAVE_DIR = Path(f'/kaggle/working/{RUN_NAME}')
SAVE_DIR.mkdir(exist_ok=True)
CHECKPOINT_PATH = SAVE_DIR / 'training_state.pth'

start_epoch = 1
best_dev_cer = float('inf')
no_improve = 0
history = []
global_step = 0

if CHECKPOINT_PATH.is_file():
    checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
    model.load_state_dict(checkpoint['model_state'])
    optimizer.load_state_dict(checkpoint['optimizer_state'])
    scheduler.load_state_dict(checkpoint['scheduler_state'])
    start_epoch = checkpoint['epoch'] + 1
    best_dev_cer = checkpoint['best_dev_cer']
    no_improve = checkpoint['no_improve']
    history = checkpoint['history']
    global_step = checkpoint.get('global_step', 0)
    print(f'Resume from epoch {start_epoch} (best CER={best_dev_cer:.4f}, step={global_step})')
else:
    print(f'Starting fresh run: {RUN_NAME}')

if device == 'cuda':
    torch.cuda.empty_cache()
gc.collect()

## 4. Boucle d'entraînement

In [ ]:
import time

for epoch in range(start_epoch, MAX_EPOCHS + 1):
    if device == 'cuda':
        torch.cuda.empty_cache()
    gc.collect()
    t0 = time.time()
    
    # Train
    model.train()
    train_loss = 0
    n_batches = 0
    
    for feats, chars, f_lens, c_lens in tqdm(train_loader, desc=f'Epoch {epoch}', leave=False):
        feats = feats.to(device, non_blocking=True)
        chars = chars.to(device, non_blocking=True)
        f_lens = f_lens.to(device, non_blocking=True)
        c_lens = c_lens.to(device, non_blocking=True)
        
        optimizer.zero_grad(set_to_none=True)
        
        with torch.amp.autocast('cuda', dtype=amp_dtype, enabled=USE_AMP and device == 'cuda'):
            log_probs = model(feats, f_lens)  # (B, T, V)
            log_probs_ctc = log_probs.transpose(0, 1)  # (T, B, V) for CTC loss
            loss = criterion(log_probs_ctc, chars, f_lens, c_lens)
        
        if scaler.is_enabled():
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()
        
        scheduler.step()
        global_step += 1
        
        train_loss += loss.item()
        n_batches += 1
    
    train_loss /= max(n_batches, 1)
    
    # Eval (greedy)
    model.eval()
    all_refs, all_hyps = [], []
    
    with torch.no_grad():
        for feats, chars, f_lens, c_lens in dev_loader:
            feats = feats.to(device, non_blocking=True)
            f_lens_dev = f_lens.to(device, non_blocking=True)
            
            with torch.amp.autocast('cuda', dtype=amp_dtype, enabled=USE_AMP and device == 'cuda'):
                log_probs = model(feats, f_lens_dev)
            preds = log_probs.argmax(dim=-1)
            
            for i in range(feats.size(0)):
                T = f_lens[i].item()
                L = c_lens[i].item()
                hyp = decode_ctc(preds[i, :T].cpu().tolist())
                ref = decode(chars[i, :L].tolist())
                all_hyps.append(hyp)
                all_refs.append(ref)
    
    dev_wer_val = wer(all_refs, all_hyps)
    dev_cer_val = cer(all_refs, all_hyps)
    lr = optimizer.param_groups[0]['lr']
    dt = time.time() - t0
    
    history.append({
        'epoch': epoch, 'train_loss': train_loss,
        'dev_wer': dev_wer_val, 'dev_cer': dev_cer_val, 'lr': lr,
        'global_step': global_step,
    })
    
    print(f'Epoch {epoch:02d} | loss={train_loss:.4f} | WER={dev_wer_val:.3f} CER={dev_cer_val:.3f} | lr={lr:.2e} | {dt:.0f}s')
    
    # Show 2 examples
    for j in range(min(2, len(all_refs))):
        print(f'  REF: {all_refs[j][:80]}')
        print(f'  HYP: {all_hyps[j][:80]}')
    
    # Save best model
    if dev_cer_val < best_dev_cer:
        best_dev_cer = dev_cer_val
        no_improve = 0
        torch.save(model.state_dict(), SAVE_DIR / 'best_asr.pt')
        print(f'  Saved best (CER={dev_cer_val:.4f})')
    else:
        no_improve += 1
    
    # Save full training state (resume-capable)
    torch.save({
        'epoch': epoch,
        'model_state': model.state_dict(),
        'optimizer_state': optimizer.state_dict(),
        'scheduler_state': scheduler.state_dict(),
        'best_dev_cer': best_dev_cer,
        'no_improve': no_improve,
        'history': history,
        'global_step': global_step,
    }, CHECKPOINT_PATH)
    
    if no_improve >= PATIENCE:
        print(f'  Early stopping at epoch {epoch}')
        break

print(f'\nBest dev CER: {best_dev_cer:.4f}')
print(f'Training state saved to {CHECKPOINT_PATH}')

## 5. Courbes d'apprentissage

In [ ]:
import matplotlib.pyplot as plt

if not history and CHECKPOINT_PATH.is_file():
    _ckpt = torch.load(CHECKPOINT_PATH, map_location='cpu')
    history = _ckpt['history']
    print(f'Loaded history from checkpoint ({len(history)} epochs)')

fig, axes = plt.subplots(1, 3, figsize=(15, 4), constrained_layout=True)

epochs_hist = [h['epoch'] for h in history]

axes[0].plot(epochs_hist, [h['train_loss'] for h in history])
axes[0].set_title('Train Loss')
axes[0].set_xlabel('Epoch')

axes[1].plot(epochs_hist, [h['dev_wer'] for h in history], label='WER')
axes[1].plot(epochs_hist, [h['dev_cer'] for h in history], label='CER')
axes[1].set_title('Dev WER / CER')
axes[1].set_xlabel('Epoch')
axes[1].legend()

axes[2].plot(epochs_hist, [h['lr'] for h in history])
axes[2].set_title('Learning Rate (warmup + cosine)')
axes[2].set_xlabel('Epoch')
axes[2].set_yscale('log')

fig.suptitle(f'DeepVox Phase 3 Conformer — {RUN_NAME}', fontsize=14)
plt.savefig(f'/kaggle/working/{RUN_NAME}_training_curves.png', dpi=150)
plt.show()

## 6. Évaluation finale sur test set

Deux décodages comparés :
1. **Greedy** : argmax frame par frame (rapide, baseline)
2. **Beam search + KenLM** : modèle de langue n-gram français

In [ ]:
# ============================================================
# 6.1 ÉVALUATION GREEDY (sans LM)
# ============================================================

# Charger le meilleur modèle
model.load_state_dict(torch.load(SAVE_DIR / 'best_asr.pt', map_location=device))
model.eval()

all_refs = []
all_hyps_greedy = []
all_logprobs = []
all_lens = []

with torch.no_grad():
    for feats, chars, f_lens, c_lens in tqdm(test_loader, desc='Test (greedy)'):
        feats = feats.to(device, non_blocking=True)
        f_lens_dev = f_lens.to(device, non_blocking=True)
        
        with torch.amp.autocast('cuda', dtype=amp_dtype, enabled=USE_AMP and device == 'cuda'):
            log_probs = model(feats, f_lens_dev)
        # Convertir en fp32 pour pyctcdecode plus tard
        log_probs = log_probs.float()
        preds = log_probs.argmax(dim=-1)
        
        for i in range(feats.size(0)):
            T = f_lens[i].item()
            L = c_lens[i].item()
            all_logprobs.append(log_probs[i, :T].cpu().numpy())
            all_lens.append(T)
            all_hyps_greedy.append(decode_ctc(preds[i, :T].cpu().tolist()))
            all_refs.append(decode(chars[i, :L].tolist()))

test_wer_greedy = wer(all_refs, all_hyps_greedy)
test_cer_greedy = cer(all_refs, all_hyps_greedy)

print(f'\n=== Test Results GREEDY ({RUN_NAME}) ===')
print(f'WER: {test_wer_greedy:.4f} ({test_wer_greedy*100:.1f}%)')
print(f'CER: {test_cer_greedy:.4f} ({test_cer_greedy*100:.1f}%)')
print(f'Samples: {len(all_refs)}')
print()

print('=== Exemples GREEDY ===')
indices = random.sample(range(len(all_refs)), min(10, len(all_refs)))
for idx in indices:
    print(f'REF: {all_refs[idx]}')
    print(f'HYP: {all_hyps_greedy[idx]}')
    print()

### 6.2 Setup KenLM

Installation pyctcdecode + téléchargement KenLM français 5-gram (~1.15 GB).

In [ ]:
# Installer pyctcdecode + KenLM (binaire pré-compilé depuis GitHub)
!pip uninstall -y pypi-kenlm 2>/dev/null
!pip install -q https://github.com/kpu/kenlm/archive/master.zip
!pip install -q pyctcdecode

try:
    import kenlm
    import pyctcdecode
    print(f'kenlm OK : {kenlm.__file__}')
    print(f'pyctcdecode OK')
except ImportError as e:
    print(f'ERROR : {e}')

import urllib.request

LM_DIR = Path('/kaggle/working/lm')
LM_DIR.mkdir(exist_ok=True)
LM_PATH = LM_DIR / 'fr_5gram.bin'

if not LM_PATH.exists():
    print('Téléchargement du KenLM français (~1 GB)...')
    url = 'https://huggingface.co/jonatasgrosman/wav2vec2-large-xlsr-53-french/resolve/main/language_model/lm.binary'
    try:
        urllib.request.urlretrieve(url, LM_PATH)
        print(f'LM téléchargé : {LM_PATH} ({os.path.getsize(LM_PATH)/1e6:.1f} MB)')
    except Exception as e:
        print(f'Échec téléchargement : {e}')
        LM_PATH = None
else:
    print(f'LM existant : {LM_PATH} ({os.path.getsize(LM_PATH)/1e6:.1f} MB)')

### 6.3 Évaluation Beam Search + KenLM

In [ ]:
from pyctcdecode import build_ctcdecoder
from deepvox.data.text import VOCAB, BLANK_IDX, UNK_IDX

# Convention pyctcdecode : 1 SEUL token vide pour le blank
labels = list(VOCAB)
labels[BLANK_IDX] = ''           # blank → chaîne vide
labels[UNK_IDX] = '⁇'            # UNK → caractère unique pour éviter doublon

assert len(labels) == len(set(labels)), f'Doublons: {labels}'
print(f'Labels ({len(labels)}): {labels[:10]}...')

decoder = build_ctcdecoder(
    labels=labels,
    kenlm_model_path=str(LM_PATH) if LM_PATH and LM_PATH.exists() else None,
    alpha=0.5,
    beta=1.0,
)
print(f'Decoder built (KenLM: {LM_PATH is not None and LM_PATH.exists()})')

In [ ]:
all_hyps_lm = []

for logp in tqdm(all_logprobs, desc='Beam search + KenLM'):
    hyp = decoder.decode(logp, beam_width=100)
    all_hyps_lm.append(hyp)

test_wer_lm = wer(all_refs, all_hyps_lm)
test_cer_lm = cer(all_refs, all_hyps_lm)

print(f'\n=== Test Results BEAM SEARCH + KenLM ({RUN_NAME}) ===')
print(f'WER: {test_wer_lm:.4f} ({test_wer_lm*100:.1f}%)')
print(f'CER: {test_cer_lm:.4f} ({test_cer_lm*100:.1f}%)')

print(f'\n=== Comparaison ===')
print(f'              Greedy   |  +KenLM   |  Δ')
print(f'WER: {test_wer_greedy*100:5.1f}%  |  {test_wer_lm*100:5.1f}%   |  {(test_wer_lm-test_wer_greedy)*100:+.1f} pp')
print(f'CER: {test_cer_greedy*100:5.1f}%  |  {test_cer_lm*100:5.1f}%   |  {(test_cer_lm-test_cer_greedy)*100:+.1f} pp')

print('\n=== Exemples comparés ===')
for idx in indices:
    print(f'REF:    {all_refs[idx]}')
    print(f'GREEDY: {all_hyps_greedy[idx]}')
    print(f'+LM:    {all_hyps_lm[idx]}')
    print()

## 7. Comparaison directe vs BiLSTM (Phase 2 Run #4)

In [ ]:
# Tableau récapitulatif Phase 2 vs Phase 3
print('=' * 70)
print(f'  COMPARAISON PHASE 2 BiLSTM vs PHASE 3 CONFORMER')
print('=' * 70)
print()
print(f'{"":24} | {"BiLSTM Run#4":>14} | {"Conformer Run#5":>16} | {"Δ":>8}')
print('-' * 70)
print(f'{"Params":24} | {"9 112 625":>14} | {f"{model.count_parameters():,}":>16} | -')
print(f'{"Taille fp32":24} | {"36.5 MB":>14} | {f"{model.count_parameters()*4/1e6:.1f} MB":>16} | -')
print(f'{"Test WER greedy":24} | {"52.8%":>14} | {f"{test_wer_greedy*100:.1f}%":>16} | {(test_wer_greedy*100-52.8):+.1f} pp')
print(f'{"Test CER greedy":24} | {"21.3%":>14} | {f"{test_cer_greedy*100:.1f}%":>16} | {(test_cer_greedy*100-21.3):+.1f} pp')
print(f'{"Test WER + KenLM":24} | {"42.6%":>14} | {f"{test_wer_lm*100:.1f}%":>16} | {(test_wer_lm*100-42.6):+.1f} pp')
print(f'{"Test CER + KenLM":24} | {"20.5%":>14} | {f"{test_cer_lm*100:.1f}%":>16} | {(test_cer_lm*100-20.5):+.1f} pp')
print()
print('Cibles Run #5 : WER greedy ≤ 38%, WER+KenLM ≤ 35%')
print('Vosk small fr 0.22 (référence taille comparable) : WER 23.95%')

In [ ]:
# Sauvegarder le modèle final
save_path = f'/kaggle/working/{RUN_NAME}_model.pt'
torch.save(model.state_dict(), save_path)
print(f'Modèle sauvegardé : {save_path}')
print(f'Params : {model.count_parameters():,}')
print(f'Taille : {os.path.getsize(save_path) / 1e6:.1f} MB')
print(f'Run: {RUN_NAME} terminé')